In [7]:
import base64
import json
import requests

In [ ]:
headers = {
    "Authorization": "Bearer ***"
}

In [4]:
# Set the collaboration ID:
#
#   - 2: Test collaboration (with IKNL and UPM)
#   - 3: IDEA4RC collaboration
#
COLLABORATION_ID = 2

In [ ]:
# Set the organization IDs that are part of this workspace. These should be the IDs of
# the vantage6 organizations:
#
#   1	- root
#   2	- ENG
#   3	- UPM
#   4	- INT
#   5	- UKE
#   6	- CLB
#   7	- FPNS
#
# These are basically all organization that are part of the workspace (thus the same
# list as in the `1-new-workspace.ipynb` notebook):
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/organization?collaboration_id={COLLABORATION_ID}",
    headers=headers
)
ORGANIZATION_IDS = [org["id"] for org in response.json()["data"]]
ORGANIZATION_IDS

[1, 3]

In [25]:
ORGANIZATION_IDS = [1]

In [26]:
# Set the session ID. This is the `v6_session` id that belongs to the RAVEN analysis.
SESSION_ID = 3

In [27]:
# When a new cohort is created, vantage6 needs to extract the data from the OMOP
# database and store it in the session as a dataframe. This is done by executing a
# vantage6 extraction task.
#
# This cell contains all the parameters that are going to end up in the `payload`
# dictionary.
#

#
# Static content
#
image = "harbor2.vantage6.ai/idea4rc/sessions:latest"
label = "omop"

#
# Dynamic content
#
# The name of the cohort, this should be unique within a session. You can probably use
# the same name that you use in the RAVEN UI. Alternatively, we can also not send it.
# In that case the name will be generated by vantage6.
# name = "Cohort_name_84"

# Each `image` can have multiple `methods`. To extract the data from the OMOP database
# we need to use the `create_cohort` method.
method = "create_cohort"

# The input for the task is the patient ids and which features we want to extract.
arguments = {
    # NOTE --- CHANGE THE PATIENT IDS TO THE PATIENT IDS OF THE COHORT ---
    # These `patient_ids` should be coming from the cohort builder in RAVEN
    "patient_ids": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    # NOTE --- CHANGE THE FEATURES TO THE FEATURES OF THE COHORT ---
    # The features are the features that we want to extract from the OMOP database.
    # This can be either "sarcoma" or "head_neck".
    # TODO The "head_neck is not implemented yet.
    "features": "sarcoma"
}

In [28]:
# before we can create a task we need to prepare task instructions. In vantage6 we can
# (but we dont in IDEA4RC) use end-to-end encryption, therefore we need to store the
# input for each organization individually.
payload = {
    "label": label,
    # "name": name, # optional, v6 will generate a name if not provided
    "task": {
        "method": method,
        "image": image,
        # In vantage6 we can (but we dont in IDEA4RC) use end-to-end encryption,
        # therefore we need to store the input for each organization individually.
        "organizations": [
            {
                "id": id_,
                "arguments": base64.b64encode(
                    json.dumps(arguments).encode("UTF-8")
                ).decode("UTF-8")
            }
            # We always create a cohort for all organizations in the study. Even though
            # in a later stage we might send computation tasks to a subset of the
            # organizations.
            for id_ in ORGANIZATION_IDS
        ]
    }
}
payload

{'label': 'omop',
 'task': {'method': 'create_cohort',
  'image': 'harbor2.vantage6.ai/idea4rc/sessions:latest',
  'organizations': [{'id': 1,
    'arguments': 'eyJwYXRpZW50X2lkcyI6IFsxLCAyLCAzLCA0LCA1LCA2LCA3LCA4LCA5LCAxMF0sICJmZWF0dXJlcyI6ICJzYXJjb21hIn0='}]}}

In [29]:
# Create a vantage6 task to extract the data from the OMOP data source and store it
# into a dataframe.
response = requests.post(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/session/{SESSION_ID}/dataframe",
    headers=headers,
    json=payload
)
TASK_ID = response.json()["last_session_task"]["id"]
DATAFRAME_ID = response.json()["id"]
response.json()

{'name': 'modest_booth',
 'session': {'id': 3,
  'link': '/server/session/3',
  'methods': ['PATCH', 'DELETE', 'GET']},
 'columns': [],
 'last_session_task': {'study': {'id': 4,
   'link': '/server/study/4',
   'methods': ['PATCH', 'DELETE', 'GET']},
  'finished_at': None,
  'databases': [{'label': 'omop',
    'type': 'source',
    'dataframe_id': None,
    'dataframe_name': None,
    'position': 0}],
  'id': 327,
  'results': '/server/result?task_id=327',
  'parent': None,
  'method': 'create_cohort',
  'session': {'id': 3,
   'link': '/server/session/3',
   'methods': ['PATCH', 'DELETE', 'GET']},
  'created_at': '2025-12-10T12:34:41.791251',
  'required_by': [],
  'children': '/server/task?parent_id=327',
  'collaboration': {'id': 2,
   'link': '/server/collaboration/2',
   'methods': ['PATCH', 'DELETE', 'GET']},
  'status': 'awaiting',
  'name': 'Session initialization (UPM Test Session 0)',
  'dataframe': {'name': 'modest_booth', 'db_label': 'omop', 'id': 89},
  'algorithm_store': 

In [42]:

# The status of the task (in this case the task that extract the data from the OMOP db
# in order to create the dataframe) can be one of the following:
#
# - pending: The task is waiting to be executed.
# - active: The task is being executed.
# - completed: The task has finished successfully.
# - crashed: The task crashed. You probably want to inspect the logs.
#
# You should poll the status of the task until it got one of the final states: crashed
# or completed
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/run?task_id={TASK_ID}",
    headers=headers,
)
response.json()
# The output also shows the status of the nodes, which can be useful to show in the
# RAVEN UI.

{'data': [{'arguments': 'eyJwYXRpZW50X2lkcyI6IFsxLCAyLCAzLCA0LCA1LCA2LCA3LCA4LCA5LCAxMF0sICJmZWF0dXJlcyI6ICJzYXJjb21hIn0=',
   'action': 'data_extraction',
   'cleanup_at': None,
   'finished_at': '2025-12-10T12:50:40.502529',
   'node': {'keycloak_client_id': '638fc96a-8954-4db4-92fa-09ff38421105',
    'name': 'Bilbao-root-node',
    'keycloak_id': '9768b637-e6bf-4f0c-b5bc-74c235856c47',
    'id': 7,
    'status': 'online'},
   'started_at': '2025-12-10T12:50:27.420708',
   'log': "LOGS of POD vantage6-run-428-kzgf6 (created by job vantage6-run-428) \n\n info > wrapper for v6-sessions\ninfo > Reading function arguments from file /app/vantage6/task/input\ninfo > Dispatching ...\n/usr/local/lib/python3.13/site-packages/v6-sessions/cohort.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.\n  im

In [44]:

# Examples
# --------
# Some example responses, at some places i've used `...` to hide details that are not
# super important for you. These example responses are similar for all tasks. That means
# that the output can also be used as example for the Summary Statistics task.
#
#
# (1) A pending task
# ------------------
# {'data': [{'status': 'pending',
#    'organization': {...},
#    'id': 39,
#    'log': None,
#    'cleanup_at': None,
#    'action': 'data_extraction',
#    'ports': [],
#    'arguments': 'eyJrd2FyZ3MiOiB7InBhdGllbnRfaWRzIjogWzEsIDIsIDMsIDQsIDUsIDYsIDcsIDgsIDksIDEwXSwgImZlYXR1cmVzIjogInNhcmNvbWEifX0=',
#    'task': {'id': 34, 'link': '/server/task/34', 'methods': ['GET', 'DELETE']},
#    'assigned_at': '2025-07-03T11:33:56.136402',
#    'finished_at': None,
#    'results': {'id': 39,
#     'link': '/server/result/39',
#     'methods': ['GET', 'PATCH']},
#    'node': {...},
#    'started_at': '2025-07-07T09:07:24.187217'}],
#  'links': {'first': '/server/run?task_id=34&page=1',
#   'self': '/server/run?task_id=34&page=1',
#   'last': '/server/run?task_id=34&page=1'}}

# (2) A running task
# ------------------
# {'data': [{'status': 'active',
#    'organization': {...},
#    'id': 41,
#    'log': None,
#    'cleanup_at': None,
#    'action': 'data_extraction',
#    'ports': [],
#    'arguments': 'eyJrd2FyZ3MiOiB7InBhdGllbnRfaWRzIjogWzEsIDIsIDMsIDQsIDUsIDYsIDcsIDgsIDksIDEwXSwgImZlYXR1cmVzIjogInNhcmNvbWEifX0=',
#    'task': {'id': 36, 'link': '/server/task/36', 'methods': ['GET', 'DELETE']},
#    'assigned_at': '2025-07-03T11:33:56.136402',
#    'finished_at': None,
#    'results': {'id': 41,
#     'link': '/server/result/41',
#     'methods': ['GET', 'PATCH']},
#    'node': {...},
#    'started_at': '2025-07-07T09:13:58.244181'}],
#  'links': {'first': '/server/run?task_id=36&page=1',
#   'self': '/server/run?task_id=36&page=1',
#   'last': '/server/run?task_id=36&page=1'}}

# (3) A completed task:
# {'data': [{'status': 'completed',
#    'organization': {...},
#    'id': 43,
#    'log': 'LOGS of POD run-43-rqbqj (created by job run-43) \n\n info > wrapper for v6-sessions\ninfo > Reading input file /app/vantage6/task/input\ninfo > Dispatching ...\n/usr/local/lib/python3.10/site-packages/v6-sessions/cohort.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.\n  import pkg_resources\ninfo > Module \'v6-sessions\' imported!\ninfo > Setting up connection to database\n{\'uri\': \'jdbc:postgresql://omop-postgres-service.datamesh.svc.cluster.local:5432/omopdb\', \'type\': \'other\'}\n$dbms\n[1] "postgresql"\n\n$extraSettings\nNULL\n\n$oracleDriver\n[1] "thin"\n\n$pathToDriver\n[1] "/usr/local/lib/python3.10/site-packages/ohdsi/database_connector/java"\n\n$user\nfunction () \nrlang::eval_tidy(userExpression)\n<bytecode: 0x5585f6f99050>\n<environment: 0x5585fca586f8>\n\n$password\nfunction () \nrlang::eval_tidy(passWordExpression)\n<bytecode: 0x5585f6f98c28>\n<environment: 0x5585fca586f8>\n\n$server\nfunction () \nrlang::eval_tidy(serverExpression)\n<bytecode: 0x5585f758bf80>\n<environment: 0x5585fca586f8>\n\n$port\nfunction () \nrlang::eval_tidy(portExpression)\n<bytecode: 0x5585f758bb58>\n<environment: 0x5585fca586f8>\n\n$connectionString\nfunction () \nrlang::eval_tidy(csExpression)\n<bytecode: 0x5585f758b730>\n<environment: 0x5585fca586f8>\n\nattr(,"class")\n[1] "ConnectionDetails"        "DefaultConnectionDetails"\n\nConnecting using PostgreSQL driver\ninfo > Retrieving variables for cohort: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]\ninfo > Loading SQL file: sarcoma\ninfo > -->  Done\ninfo > Injecting patient IDs into SQL\ninfo > -->  Done\ninfo > Executing SQL\ninfo > Converting dataframe to pandas\ninfo > -->  Done\ninfo > Done!\ninfo > Writing output to /app/vantage6/task/output\n\x1b[?25hR[write to console]: Warning messages:\n\nR[write to console]: 1: \nR[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :\nR[write to console]: \n \nR[write to console]:  library ‘/usr/lib/R/site-library’ contains no packages\n\nR[write to console]: 2: \nR[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :\nR[write to console]: \n \nR[write to console]:  library ‘/usr/lib/R/site-library’ contains no packages\n\nR[write to console]: 3: \nR[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :\nR[write to console]: \n \nR[write to console]:  library ‘/usr/lib/R/site-library’ contains no packages\n\nR[write to console]: 4: \nR[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :\nR[write to console]: \n \nR[write to console]:  library ‘/usr/lib/R/site-library’ contains no packages\n\nR[write to console]: 5: \nR[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :\nR[write to console]: \n \nR[write to console]:  library ‘/usr/lib/R/site-library’ contains no packages\n\n\x1b[0m \n\n\n',
#    'cleanup_at': None,
#    'action': 'data_extraction',
#    'ports': [],
#    'arguments': 'eyJrd2FyZ3MiOiB7InBhdGllbnRfaWRzIjogWzEsIDIsIDMsIDQsIDUsIDYsIDcsIDgsIDksIDEwXSwgImZlYXR1cmVzIjogInNhcmNvbWEifX0=',
#    'task': {'id': 38, 'link': '/server/task/38', 'methods': ['GET', 'DELETE']},
#    'assigned_at': '2025-07-03T11:33:56.136402',
#    'finished_at': '2025-07-07T09:33:56.195070',
#    'results': {'id': 43,
#     'link': '/server/result/43',
#     'methods': ['GET', 'PATCH']},
#    'node': {...},
#    'started_at': '2025-07-07T09:32:57.230257'}],
#  'links': {'first': '/server/run?task_id=38&page=1',
#   'self': '/server/run?task_id=38&page=1',
#   'last': '/server/run?task_id=38&page=1'}}

# (4) A crashed task
# ------------------
# {'data': [{'status': 'crashed',
#    'organization': {...},
#    'id': 39,
#    'log': 'LOGS of POD run-39-p9tkc (created by job run-39) \n\n info > wrapper for v6-sessions\ninfo > Reading input file /app/vantage6/task/input\ninfo > Dispatching ...\n/usr/local/lib/python3.10/site-packages/v6-sessions/cohort.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.\n  import pkg_resources\ninfo > Module \'v6-sessions\' imported!\ninfo > Setting up connection to database\n{\'uri\': \'jdbc:postgresql://omop-postgres-service.datamesh.svc.cluster.local:5432/omopdb\', \'type\': \'other\'}\n$dbms\n[1] "postgresql"\n\n$extraSettings\nNULL\n\n$oracleDriver\n[1] "thin"\n\n$pathToDriver\n[1] "/usr/local/lib/python3.10/site-packages/ohdsi/database_connector/java"\n\n$user\nfunction () \nrlang::eval_tidy(userExpression)\n<bytecode: 0x55c12b738f70>\n<environment: 0x55c1313a55c8>\n\n$password\nfunction () \nrlang::eval_tidy(passWordExpression)\n<bytecode: 0x55c12b738b48>\n<environment: 0x55c1313a55c8>\n\n$server\nfunction () \nrlang::eval_tidy(serverExpression)\n<bytecode: 0x55c12b8a5a80>\n<environment: 0x55c1313a55c8>\n\n$port\nfunction () \nrlang::eval_tidy(portExpression)\n<bytecode: 0x55c12b8a5658>\n<environment: 0x55c1313a55c8>\n\n$connectionString\nfunction () \nrlang::eval_tidy(csExpression)\n<bytecode: 0x55c12b8a5230>\n<environment: 0x55c1313a55c8>\n\nattr(,"class")\n[1] "ConnectionDetails"        "DefaultConnectionDetails"\n\nConnecting using PostgreSQL driver\ninfo > Retrieving variables for cohort: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]\ninfo > Loading SQL file: sarcoma\ninfo > -->  Done\ninfo > Injecting patient IDs into SQL\ninfo > -->  Done\ninfo > Executing SQL\ninfo > Converting dataframe to pandas\n   PATIENT_ID  ...  N_CANCER_EPISODES\n1        10.0  ...                0.0\n2         2.0  ...                0.0\n3         5.0  ...                0.0\n4         8.0  ...                0.0\n5         6.0  ...                0.0\n\n[5 rows x 21 columns]\ninfo > -->  Done\ninfo > Done!\ninfo > Writing output to /app/vantage6/task/output\nTraceback (most recent call last):\n  File "<string>", line 1, in <module>\n  File "/usr/local/lib/python3.10/site-packages/vantage6/algorithm/tools/wrap.py", line 75, in wrap_algorithm\n    _write_output(output, output_file)\n  File "/usr/local/lib/python3.10/site-packages/vantage6/algorithm/tools/wrap.py", line 199, in _write_output\n    pq.write_table(output, output_file)\n  File "/usr/local/lib/python3.10/site-packages/pyarrow/parquet/core.py", line 1884, in write_table\n    where, table.schema,\nAttributeError: \'bytes\' object has no attribute \'schema\'\n\x1b[?25hR[write to console]: Warning messages:\n\nR[write to console]: 1: \nR[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :\nR[write to console]: \n \nR[write to console]:  library ‘/usr/lib/R/site-library’ contains no packages\n\nR[write to console]: 2: \nR[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :\nR[write to console]: \n \nR[write to console]:  library ‘/usr/lib/R/site-library’ contains no packages\n\nR[write to console]: 3: \nR[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :\nR[write to console]: \n \nR[write to console]:  library ‘/usr/lib/R/site-library’ contains no packages\n\nR[write to console]: 4: \nR[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :\nR[write to console]: \n \nR[write to console]:  library ‘/usr/lib/R/site-library’ contains no packages\n\nR[write to console]: 5: \nR[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :\nR[write to console]: \n \nR[write to console]:  library ‘/usr/lib/R/site-library’ contains no packages\n\n\x1b[0m \n\n\n',
#    'cleanup_at': None,
#    'action': 'data_extraction',
#    'ports': [],
#    'arguments': 'eyJrd2FyZ3MiOiB7InBhdGllbnRfaWRzIjogWzEsIDIsIDMsIDQsIDUsIDYsIDcsIDgsIDksIDEwXSwgImZlYXR1cmVzIjogInNhcmNvbWEifX0=',
#    'task': {'id': 34, 'link': '/server/task/34', 'methods': ['GET', 'DELETE']},
#    'assigned_at': '2025-07-03T11:33:56.136402',
#    'finished_at': '2025-07-07T09:07:55.422012',
#    'results': {'id': 39,
#     'link': '/server/result/39',
#     'methods': ['GET', 'PATCH']},
#    'node': {...},
#    'started_at': '2025-07-07T09:07:24.187217'}],
#  'links': {'first': '/server/run?task_id=34&page=1',
#   'self': '/server/run?task_id=34&page=1',
#   'last': '/server/run?task_id=34&page=1'}}

In [45]:
# This session ID should be stored in the RAVEN database in the ?? table
# (`v6_dataframe`).
DATAFRAME_ID

89